# 📓 Notebook 09 — Memory---**Phase 4 · Agent Architecture**> An agent without memory forgets everything after each conversation. Memory transforms a chatbot into a persistent assistant.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Short-term memory | Conversation buffer within a session || Long-term memory | Persistent vector-stored memories || Summarization | Compress old context to save tokens || Memory architectures | Buffer, summary, vector, hybrid |### 🏗️ Build: Persistent Personal Assistant

## 1. Setup

In [ ]:
import os, sys
import litellm

from setup_llm import DEFAULT_MODEL, is_proxy_mode, DEFAULT_EMBEDDING

mode = "🔗 proxy" if is_proxy_mode() else "🔑 direct"
print(f"🤖 Model: {DEFAULT_MODEL} ({mode})")

---## 2. Memory Architectures Overview### The Four Types of Agent Memory| Type | What It Stores | Persistence | Token Cost | Use Case ||------|---------------|-------------|------------|----------|| **Buffer** | Full conversation history | Session only | Grows linearly | Short conversations || **Window** | Last N messages | Session only | Fixed | Long conversations || **Summary** | Compressed summary of conversation | Session or persistent | Low, fixed | Very long conversations || **Vector** | Embedded conversation turns | Persistent | Retrieval cost | Cross-session memory |> **Interview Tip:** *"How would you implement memory for a production chatbot?"*  > Hybrid approach: Window buffer for current session + Vector store for long-term + Summary for context handoff.

---## 3. Buffer Memory (Short-term)

In [ ]:
class BufferMemory:    """Full conversation history — simplest memory type."""        def __init__(self, max_messages=50):        self.messages = []        self.max_messages = max_messages        def add(self, role, content):        self.messages.append({            "role": role, "content": content,            "timestamp": datetime.now().isoformat()        })        # Trim if too long        if len(self.messages) > self.max_messages:            self.messages = self.messages[-self.max_messages:]        def get_messages(self):        return [{"role": m["role"], "content": m["content"]} for m in self.messages]        def clear(self):        self.messages = []        def token_estimate(self):        return sum(len(m["content"]) // 4 for m in self.messages)        def __repr__(self):        return f"BufferMemory({len(self.messages)} messages, ~{self.token_estimate()} tokens)"# Demomem = BufferMemory()mem.add("user", "My name is Abhishek")mem.add("assistant", "Nice to meet you, Abhishek!")mem.add("user", "I work in fintech")mem.add("assistant", "That's a great field! Fintech is booming.")print(f"📦 {mem}")print(f"   Messages: {mem.get_messages()}")

---## 4. Window Memory

In [ ]:
class WindowMemory:    """Keeps only the last N messages — prevents unbounded growth."""        def __init__(self, window_size=10):        self.all_messages = []        self.window_size = window_size        def add(self, role, content):        self.all_messages.append({"role": role, "content": content})        def get_messages(self):        return self.all_messages[-self.window_size:]        def __repr__(self):        return f"WindowMemory({len(self.all_messages)} total, showing last {self.window_size})"win_mem = WindowMemory(window_size=4)for i in range(10):    win_mem.add("user", f"Message {i}")    win_mem.add("assistant", f"Response {i}")print(f"📦 {win_mem}")print(f"   Visible: {[m['content'] for m in win_mem.get_messages()]}")

---## 5. Summary Memory

In [ ]:
class SummaryMemory:    """Compresses old conversation into a running summary."""        def __init__(self, model=None, summarize_after=10):        self.model = model or DEFAULT_MODEL        self.messages = []        self.summary = ""        self.summarize_after = summarize_after        def add(self, role, content):        self.messages.append({"role": role, "content": content})                if len(self.messages) >= self.summarize_after:            self._summarize()        def _summarize(self):        """Compress messages into summary."""        conversation = "\n".join([f"{m['role']}: {m['content']}" for m in self.messages])                prompt = f"""Summarize this conversation in 2-3 sentences, preserving all key facts, preferences, and decisions mentioned:Previous summary: {self.summary if self.summary else 'None'}Recent conversation:{conversation}Updated summary:"""                response = litellm.completion(            model=self.model,            messages=[{"role": "user", "content": prompt}],            temperature=0, max_tokens=200,        )                self.summary = response.choices[0].message.content        self.messages = self.messages[-2:]  # Keep last 2 for continuity        print(f"  📝 Summarized! New summary: {self.summary[:80]}...")        def get_messages(self):        msgs = []        if self.summary:            msgs.append({"role": "system", "content": f"Conversation summary so far: {self.summary}"})        msgs.extend([{"role": m["role"], "content": m["content"]} for m in self.messages])        return msgs# Demosum_mem = SummaryMemory(summarize_after=6)conversations = [    ("user", "Hi, I'm Abhishek. I work in fintech."),    ("assistant", "Hello Abhishek! Great to hear you work in fintech."),    ("user", "I'm learning about AI agents for trading systems."),    ("assistant", "That's a great application! Trading agents can automate analysis and execution."),    ("user", "I prefer Python over Java for this."),    ("assistant", "Python is excellent for AI trading systems."),    ("user", "What framework should I use?"),]for role, content in conversations:    sum_mem.add(role, content)print(f"\n📦 Summary Memory state:")print(f"   Summary: {sum_mem.summary}")print(f"   Recent messages: {len(sum_mem.messages)}")

---## 6. Vector Memory (Long-term)

In [ ]:
class VectorMemory:    """Long-term memory backed by vector similarity search."""        def __init__(self, embedding_model=None):        self.emb_model = embedding_model or DEFAULT_EMBEDDING        self.memories = []      # {"text": ..., "role": ..., "timestamp": ..., "embedding": ...}        def store(self, text, role="user", metadata=None):        """Store a memory with its embedding."""        embedding = get_embedding(text)        memory = {            "text": text,            "role": role,            "timestamp": datetime.now().isoformat(),            "embedding": embedding,            "metadata": metadata or {},        }        self.memories.append(memory)        def recall(self, query, top_k=3):        """Retrieve most relevant memories."""        if not self.memories:            return []                query_emb = get_embedding(query)        scored = []        for i, mem in enumerate(self.memories):            score = cosine_similarity(query_emb, mem["embedding"])            scored.append((i, score))                scored.sort(key=lambda x: x[1], reverse=True)                return [{            "text": self.memories[idx]["text"],            "role": self.memories[idx]["role"],            "score": round(score, 4),            "timestamp": self.memories[idx]["timestamp"],        } for idx, score in scored[:top_k]]        def save(self, path):        """Persist memories to disk."""        data = [{k: v for k, v in m.items() if k != "embedding"} for m in self.memories]        embeddings = [m["embedding"] for m in self.memories]        with open(path, "w") as f:            json.dump({"memories": data, "embeddings": embeddings}, f)        print(f"  💾 Saved {len(self.memories)} memories to {path}")        def load(self, path):        """Load memories from disk."""        with open(path) as f:            data = json.load(f)        self.memories = []        for mem, emb in zip(data["memories"], data["embeddings"]):            mem["embedding"] = emb            self.memories.append(mem)        print(f"  📂 Loaded {len(self.memories)} memories")        def __repr__(self):        return f"VectorMemory({len(self.memories)} memories)"# Demovmem = VectorMemory()vmem.store("My name is Abhishek and I work in fintech", "user")vmem.store("I'm building an AI trading system", "user")vmem.store("I prefer Python for machine learning", "user")vmem.store("I'm interested in quantitative finance", "user")vmem.store("I have experience with ETL pipelines", "user")vmem.store("I'm learning about LangChain and agents", "user")# Recall relevant memoriesprint("🧠 Vector Memory Recall")print("=" * 50)for query in ["What programming language?", "Tell me about their work", "What are they learning?"]:    results = vmem.recall(query, top_k=2)    print(f"\n🔍 \"{query}\"")    for r in results:        print(f"   [{r['score']:.3f}] {r['text']}")

---## 📝 Interview Quiz — Memory### Q1: What are the four types of agent memory? Compare them.<details><summary>💡 Answer</summary>| Type | Storage | Persistence | Token Cost | Best For ||------|---------|-------------|------------|----------|| **Buffer** | Full history | Session | O(n) growing | Short conversations || **Window** | Last N messages | Session | O(1) fixed | Long sessions || **Summary** | Compressed summary | Can persist | O(1) fixed | Very long conversations || **Vector** | Embedded memories | Persistent | O(k) per recall | Cross-session, personalization |**Production recommendation:** Hybrid — Window for current session + Vector for long-term + Summary for handoff.</details>### Q2: How do you handle the growing context problem?<details><summary>💡 Answer</summary>As conversation grows, token count grows → hits context window limits and increases cost.Solutions:1. **Window memory** — Only keep last N messages2. **Summarization** — Compress old messages periodically3. **Vector memory** — Store everything, retrieve only relevant bits4. **Hierarchical** — Keep recent messages in full, older ones as summaries5. **Token budget** — Allocate fixed budget: 30% system prompt, 20% memory, 50% current turn</details>### Q3: How would you implement "the agent remembers user preferences across sessions"?<details><summary>💡 Answer</summary>1. **Detect preferences** — Use LLM to extract key facts/preferences from conversation2. **Store in vector DB** — Embed and store with metadata (user_id, timestamp, category)3. **Recall on new session** — When user starts new conversation, retrieve relevant memories4. **Inject into system prompt** — "User preferences: {recalled_memories}"5. **Update periodically** — If preferences change, update vector entriesKey: Extract structured "memory objects" (not raw conversation), categorize them (preference, fact, instruction), and make them searchable.</details>### Q4: What is conversation summarization? When does it lose important information?<details><summary>💡 Answer</summary>Summarization compresses a full conversation into a short summary using the LLM itself.**Loses information when:**- Specific numbers/dates are mentioned but not preserved in summary- Subtle context or implications are compressed away- User ambiguity is resolved incorrectly in summary- Multiple topics discussed — summary may drop minor topics**Mitigation:** Include "preserve key facts, numbers, and decisions" in summarization prompt. Use fact-extraction alongside summarization.</details>

---## 🏗️ BUILD: Persistent Personal Assistant

In [ ]:
class PersonalAssistant:    """A persistent personal assistant with hybrid memory."""        def __init__(self, name="Assistant", model=None):        self.name = name        self.model = model or DEFAULT_MODEL        self.buffer = BufferMemory(max_messages=20)        self.long_term = VectorMemory()        self.user_profile = {}        def _extract_facts(self, text):        """Extract key facts from user messages for long-term storage."""        response = litellm.completion(            model=self.model,            messages=[{                "role": "user",                "content": f"Extract key personal facts from this message. Return JSON with 'facts' array. If no personal facts, return empty array.\n\nMessage: \"{text}\""            }],            response_format={"type": "json_object"},            temperature=0, max_tokens=200,        )        try:            data = json.loads(response.choices[0].message.content)            return data.get("facts", [])        except:            return []        def chat(self, user_message, verbose=True):        """Chat with memory."""        # Store in buffer        self.buffer.add("user", user_message)                # Extract and store facts for long-term memory        facts = self._extract_facts(user_message)        for fact in facts:            if isinstance(fact, str) and len(fact) > 5:                self.long_term.store(fact, "fact")                if verbose:                    print(f"  🧠 Stored fact: {fact}")                # Recall relevant long-term memories        relevant_memories = self.long_term.recall(user_message, top_k=3) if self.long_term.memories else []                # Build context        memory_context = ""        if relevant_memories:            memory_texts = [m["text"] for m in relevant_memories if m["score"] > 0.3]            if memory_texts:                memory_context = f"\n\nRelevant things you know about the user: {'; '.join(memory_texts)}"                system_prompt = f"""You are {self.name}, a helpful personal assistant.You remember things the user tells you and use that context to be more helpful.Be conversational and reference what you know about the user when relevant.{memory_context}"""                messages = [{"role": "system", "content": system_prompt}]        messages.extend(self.buffer.get_messages())                response = litellm.completion(            model=self.model, messages=messages,            temperature=0.7, max_tokens=500,        )                reply = response.choices[0].message.content        self.buffer.add("assistant", reply)                return reply        def status(self):        print(f"📊 {self.name} Status:")        print(f"   Buffer: {len(self.buffer.messages)} messages")        print(f"   Long-term: {len(self.long_term.memories)} memories")# Demo the personal assistantpa = PersonalAssistant("Jarvis")conversations = [    "Hi! I'm Abhishek. I work as a software engineer in fintech.",    "I'm currently learning about AI agents and RAG systems.",    "My favorite programming language is Python, but I also use C# for .NET projects.",    "Can you remind me what field I work in?",    "What programming languages do I use?",]print("🤖 Personal Assistant Demo")print("=" * 60)for msg in conversations:    print(f"\n👤 {msg}")    reply = pa.chat(msg)    print(f"🤖 {reply}")pa.status()

---## ✅ Notebook 09 Summary| Concept | Key Takeaway ||---------|-------------|| Buffer memory | Full history, simple, grows unbounded || Window memory | Last N messages, fixed cost || Summary memory | Compressed, token-efficient, may lose detail || Vector memory | Persistent, searchable, selective recall || Hybrid approach | Window + Vector + Summary for production || Fact extraction | LLM extracts key facts for long-term storage |### ➡️ Next: [Notebook 10 — Planning Agents](./10_planning_agent.ipynb)